In [0]:
data = [
(1,"John","Pune"),
(2,"Sara","Mumbai"),
(3,"Mike","Delhi")
]

columns = ["id","name","city"]

df = spark.createDataFrame(data,columns)

df.write.format("delta").mode("overwrite").saveAsTable("customers")

In [0]:
%sql
select * from customers

In [0]:
cdc_data = [
(1,"John","Bangalore","UPDATE"),
(4,"Alice","Chennai","INSERT"),
(2,"Sara","Mumbai","DELETE")
]

cdc_columns = ["id","name","city","operation"]

cdc_df = spark.createDataFrame(cdc_data,cdc_columns)

cdc_df.createOrReplaceTempView("cdc_table")

In [0]:
%sql
select * from cdc_table

In [0]:
%sql
MERGE INTO customers AS target 
USING cdc_table AS source
ON target.id = source.id 

WHEN MATCHED AND source.operation = 'UPDATE'
THEN UPDATE SET
target.name = source.name,
target.city = source.city

WHEN MATCHED AND source.operation = 'DELETE'
THEN DELETE

WHEN NOT MATCHED AND source.operation = 'INSERT'
THEN INSERT (id, name, city)
VALUES (source.id, source.name, source.city)



In [0]:
%sql
select * from customers